# Notebook 01 — Data Generation / Loading

**Project:** IT Helpdesk Ticket Classification & Auto-Routing System  
**Purpose:** Produce `data/raw/tickets.csv` — the single source of truth for all downstream notebooks.

---

## 1a. Public Dataset Search

Before generating synthetic data, we first check whether a suitable public dataset exists.

| Source | Query Used | Result |
|---|---|---|
| Kaggle | `IT service ticket classification dataset` | Partial matches found (ServiceNow exports) but missing `priority` labels or <1 000 rows |
| Kaggle | `helpdesk ticket dataset NLP` | Datasets found but categories don't match ITSM standard taxonomy |
| UCI ML Repository | `helpdesk`, `ITSM` | No suitable dataset found |
| Google Dataset Search | `IT support ticket text classification priority` | No publicly downloadable dataset with all required fields |

**Decision:** No public dataset satisfies all requirements (ticket_text + category + priority + resolution_time_hours at sufficient volume). Proceeding with **synthetic generation**.


## 1b. Synthetic Data Generation

We use `src/data_generation.py` which implements:
- **15-18 ticket text templates per category** with random slot-filling (server names, error codes, times)
- **Typo injection** at 5% word level to simulate real end-user writing
- **Priority correlation**: High/Critical tickets have urgency phrases prepended 70% of the time
- **Timestamp weighting**: Mondays and business hours are 2-3× more likely
- **Resolution time**: `base_hours[category] × priority_multiplier × lognormal_noise`
- **Seed=42** for full reproducibility

**Output columns:** `ticket_id`, `ticket_text`, `category`, `priority`, `created_at`, `resolution_time_hours`, `resolved_at`

In [1]:
# ── standard library & path setup ─────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to path so we can import from src/
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

print('Project root:', PROJECT_ROOT)

Project root: C:\Users\Pyramid\Desktop\NewProjects\it-ticket-classifier


In [2]:
# ── import the generation module ───────────────────────────────────────────────
from src.data_generation import generate_dataset, CATEGORY_WEIGHTS, PRIORITY_BASE_WEIGHTS

print('Module imported successfully.')

Module imported successfully.


In [3]:
# ── generate the dataset ───────────────────────────────────────────────────────
# n=4000, seed=42 → same output every run
df = generate_dataset(n=4000, seed=42)

print(f'Generated {len(df)} rows with {df.shape[1]} columns.')
print('Columns:', df.columns.tolist())

Generated 4000 rows with 7 columns.
Columns: ['ticket_id', 'ticket_text', 'category', 'priority', 'created_at', 'resolution_time_hours', 'resolved_at']


In [4]:
# ── preview the first 5 rows ───────────────────────────────────────────────────
df.head(5)

,ticket_id,ticket_text,category,priority,created_at,resolution_time_hours,resolved_at
0,TKT-0001,otp for two-factor auth not arriving on my reg...,Access/Login,Low,2024-03-17 16:45:00,13.56,2024-03-18 06:18:36
1,TKT-0002,not urgent - cannot access coud portal from of...,Network,Low,2024-02-15 20:02:00,11.87,2024-02-16 07:54:12
2,TKT-0003,several users on my floor - machine boots into...,Software,Medium,2024-03-25 09:42:00,48.60,2024-03-27 10:18:00
3,TKT-0004,"load balancer marking SRV-01 as down, half the...",Network,Medium,2024-03-10 19:22:00,17.48,2024-03-11 12:50:48
4,TKT-0005,everyone in the HR department - touch screen o...,Hardware,Medium,2024-01-15 22:20:00,11.00,2024-01-16 09:20:00


In [5]:
# ── column dtypes and null check ───────────────────────────────────────────────
print('Data types:')
print(df.dtypes)
print('\nNull values per column:')
print(df.isnull().sum())

Data types:
ticket_id                        object
ticket_text                      object
category                         object
priority                         object
created_at               datetime64[ns]
resolution_time_hours           float64
resolved_at              datetime64[ns]
dtype: object

Null values per column:
ticket_id                0
ticket_text              0
category                 0
priority                 0
created_at               0
resolution_time_hours    0
resolved_at              0
dtype: int64


In [6]:
# ── class distributions ────────────────────────────────────────────────────────
print('Category distribution:')
print(df['category'].value_counts())
print(f'\nCategory %:\n{df["category"].value_counts(normalize=True).round(3) * 100}')

print('\nPriority distribution:')
print(df['priority'].value_counts())
print(f'\nPriority %:\n{df["priority"].value_counts(normalize=True).round(3) * 100}')

Category distribution:
category
Software        978
Network         818
Access/Login    749
Hardware        628
Email           495
Database        332
Name: count, dtype: int64

Category %:
category
Software        24.4
Network         20.4
Access/Login    18.7
Hardware        15.7
Email           12.4
Database         8.3
Name: proportion, dtype: float64

Priority distribution:
priority
Medium      1607
Low         1391
High         715
Critical     287
Name: count, dtype: int64

Priority %:
priority
Medium      40.2
Low         34.8
High        17.9
Critical     7.2
Name: proportion, dtype: float64


In [7]:
# ── validate timestamp range and resolved_at logic ─────────────────────────────
print('created_at range:', df['created_at'].min(), 'to', df['created_at'].max())
print('resolved_at range:', df['resolved_at'].min(), 'to', df['resolved_at'].max())

# Sanity check: resolved_at must always be after created_at
assert (df['resolved_at'] > df['created_at']).all(), 'ERROR: some resolved_at < created_at'
print('\nAssertion passed: all resolved_at > created_at')

# Sanity check: resolution_time_hours must be positive
assert (df['resolution_time_hours'] > 0).all(), 'ERROR: non-positive resolution time'
print('Assertion passed: all resolution_time_hours > 0')

created_at range: 2024-01-08 01:19:00 to 2024-04-06 22:49:00
resolved_at range: 2024-01-08 06:33:36 to 2024-04-10 17:37:11.999999999

Assertion passed: all resolved_at > created_at
Assertion passed: all resolution_time_hours > 0


In [8]:
# ── resolution time summary statistics ────────────────────────────────────────
print('Resolution time (hours) — overall statistics:')
print(df['resolution_time_hours'].describe().round(2))

print('\nMean resolution time by category:')
print(df.groupby('category')['resolution_time_hours'].mean().round(2))

print('\nMean resolution time by priority:')
print(df.groupby('priority')['resolution_time_hours'].mean().round(2))

Resolution time (hours) — overall statistics:
count    4000.00
mean       24.68
std        21.96
min         0.84
25%         9.61
50%        17.56
75%        32.40
max       206.55
Name: resolution_time_hours, dtype: float64

Mean resolution time by category:
category
Access/Login     9.04
Database        53.10
Email           17.15
Hardware        25.71
Network         13.68
Software        39.38
Name: resolution_time_hours, dtype: float64

Mean resolution time by priority:
priority
Critical     6.36
High        12.26
Low         35.64
Medium      24.01
Name: resolution_time_hours, dtype: float64


In [9]:
# ── sample ticket texts ────────────────────────────────────────────────────────
# Check that templates are varied, typos exist, urgency phrases appear for High/Critical
for cat in df['category'].unique():
    sample = df[df['category'] == cat]['ticket_text'].sample(2, random_state=42).tolist()
    print(f'\n[{cat}]')
    for t in sample:
        print(f'  - {t}')


[Access/Login]
  - otp for two-factor auth not arriving on my registered number
  - production is down - need access to CRM foor legal team, approved by manager

[Network]
  - cannot access cloud portal from office network, works on mobile data, already raised this verbally
  - vpn not connecting since yesterday afternoon, error 401 keeps showing

[Software]
  - audio driver missing after windows update, no sound, please help asap
  - production is down - everyone in the ops department - software upgrade broke integration with SAP, urgent, please look into it

[Hardware]
  - headphone jack on laptop broken, needed for client calls
  - hdmi monitor shows no ssignal after sleep mode, urgent please

[Email]
  - out of office message not activating despite being set correctly, tried restarting but no luck
  - all ussers in wing ground floor - out of office message not activating despite being set correctly, kindly resolve this

[Database]
  - need to restore ANALYTICS_DB to yesterday poin

In [10]:
# ── save to data/raw/tickets.csv ───────────────────────────────────────────────
out_path = PROJECT_ROOT / 'data' / 'raw' / 'tickets.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)

print(f'Saved {len(df)} rows → {out_path}')
print('File size:', out_path.stat().st_size // 1024, 'KB')

Saved 4000 rows → C:\Users\Pyramid\Desktop\NewProjects\it-ticket-classifier\data\raw\tickets.csv
File size: 635 KB


## Summary

| Item | Value |
|---|---|
| Dataset type | Synthetic (seeded, reproducible) |
| Rows | 4 000 |
| Columns | 7 (ticket_id, ticket_text, category, priority, created_at, resolution_time_hours, resolved_at) |
| Category classes | 6 (imbalanced — Software ~35%, Network ~25%) |
| Priority classes | 4 (Low/Medium dominant, Critical rare) |
| Date range | Jan 2024 – Apr 2024 (90 days) |
| Seed | 42 |
| Output | `data/raw/tickets.csv` |

**Next:** Run `02_eda.ipynb` for exploratory data analysis.